In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import argparse
from tqdm import tqdm
tqdm.pandas()
import re
import gc

In [2]:
df = pd.read_csv("/kaggle/input/final-data/speeches_all_emi.csv")
df2 = pd.read_csv("/kaggle/input/german-parliament/speeches_all.csv")

/tmp/ipykernel_17/544387983.py:1: DtypeWarning: Columns (7,8,11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/kaggle/input/final-data/speeches_all_emi.csv")
/tmp/ipykernel_17/544387983.py:2: DtypeWarning: Columns (7,8,11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv("/kaggle/input/german-parliament/speeches_all.csv")


In [3]:
gc.collect()
def config(parser):  
    parser.add_argument('--model_name_or_path', default='/kaggle/input/word2vec-new/word2vec_new.model')
    parser.add_argument('--input_file', default='/kaggle/input/parliament2/speeches_clean.csv')
    parser.add_argument('--output_file', default='speeches_new_emi.csv')
    parser.add_argument('--evidence_lexicon', default='/kaggle/input/dictionary2/PRODEMINFO_German_keywords.csv')
    parser.add_argument('--intuition_lexicon', default='/kaggle/input/dictionary2/PRODEMINFO_German_keywords.csv')
    parser.add_argument('--save_embeddings', action="store_true")
    parser.add_argument('--smoke_test', action="store_true")
    parser.add_argument('--text_column', type=str, default='speechContent')
    parser.add_argument('--compression_type', type=str, default='infer')
    parser.add_argument('--length_threshold', type=int, default=10)
    parser.add_argument('--tab_delimiter', action="store_true")
    parser.add_argument('--chunk_text', action="store_true")
    parser.add_argument('--min_chunk_length', type=int, default=50)
    parser.add_argument('--max_chunk_length', type=int, default=150)
    parser.add_argument('--id_column', type=str, default="id")
    parser.add_argument('--cap_zero', action="store_true", help="If set, caps negative similarity values to zero.")
    return parser 


# Set up arguments manually
class Args:
    model_name_or_path = '/kaggle/input/sbert-model-new/model'
    input_file = '/kaggle/input/german-parliament/speeches_all.csv'
    output_file = 'speeches_all_emi.csv'
    evidence_lexicon = '/kaggle/input/dictionary2/PRODEMINFO_German_keywords.csv'
    intuition_lexicon = '/kaggle/input/dictionary2/PRODEMINFO_German_keywords.csv'
    save_embeddings = False
    smoke_test = False
    text_column = 'sentence'
    compression_type = 'infer'
    length_threshold = 10
    tab_delimiter = False
    chunk_text = True
    min_chunk_length = 50
    max_chunk_length = 150
    id_column = "id"
    cap_zero = False  # Set this to True to cap negative similarity values at zero


args = Args()


In [4]:
# Load stopwords
with open("/kaggle/input/final-data/de_stopwords.txt", 'r', encoding='utf-8') as file:
    stopwords_set = {line.strip() for line in file if line.strip()}

# Preprocessing Functions
def remove_special_characters_rm(text):
    """Remove special characters but mark them with placeholders."""
    text = str(text)
    pattern = r'[^a-zA-Z0-9äöüÄÖÜß\s]'
    return re.sub(pattern, lambda x: f"<REMOVED>", text)

def remove_stopwords_rm(text, stopwords_set):
    """Remove stopwords but mark them with placeholders."""
    mostly_numeric_pattern = r"^\d*[a-zA-Z]?\d*$"
    return ' '.join(
        [word if word not in stopwords_set and len(word) > 2 and not re.match(mostly_numeric_pattern, word) 
         else f"<REMOVED:{word}>" 
         for word in [re.sub(r'<REMOVED>', '', x) for x in text.split()]]
    )

def remove_special_characters(text):
    pattern = r'[^a-zA-Z0-9äöüÄÖÜß\s]'
    clean_text = re.sub(pattern, '', text)
    return clean_text

def remove_stopwords(text, stopwords_set):
    mostly_numeric_pattern = r"^\d*[a-zA-Z]?\d*$"
    return ' '.join([word for word in text.split() if word not in stopwords_set and len(word) > 2 and not re.match(mostly_numeric_pattern, word)])

# Chunking Function
def chunk_preprocessed_and_original_with_mapping(preprocessed_text, original_text, args):
    """
    Chunk preprocessed text and align the original text while ignoring placeholders in the range calculation.
    
    Parameters:
    - preprocessed_text (str): Preprocessed (cleaned) text with placeholders.
    - original_text (str): Original unprocessed text.
    - args: Object with `max_chunk_length` and `min_chunk_length`.
    
    Returns:
    - processed_chunks (list): Clean chunks of the preprocessed text (placeholders removed).
    - original_chunks (list): Aligned chunks of the original text.
    """
    # Function to check the length of preprocessed words
    def check_length(preprocessed_words, max_chunk_length):
        # Exclude words that start with "<REMOVED:" and ensure the condition for the last word
        filtered_words = [
            word for word in preprocessed_words
            if not word.startswith("<REMOVED:") and word != "<REMOVED>"
        ]
        
        # Check if the length of filtered words is within the max chunk length
        if len(filtered_words) <= max_chunk_length:
            return True
        return False

    
    max_chunk_length = args.max_chunk_length
    min_chunk_length = args.min_chunk_length


    #preprocessed_text = 
    preprocessed_words = str(preprocessed_text).split()
    original_words = str(original_text).split()

    #if check_length(preprocessed_words, max_chunk_length): 
    #    processed_chunks = [" ".join(preprocessed_words)]
    #    original_chunks = [" ".join(original_words)]
    #    return processed_chunks, original_chunks
        

    processed_chunks = []
    original_chunks = []

    start_idx = 0
    original_idx = 0

    while start_idx < len(preprocessed_words):
        # Define the chunk range, skipping placeholders
        chunk_length = 0
        chunk_end_idx = start_idx
        while chunk_end_idx < len(preprocessed_words) and chunk_length < max_chunk_length:
            if not preprocessed_words[chunk_end_idx].startswith("<REMOVED:") and preprocessed_words[chunk_end_idx] != "<REMOVED>":
                chunk_length += 1
            chunk_end_idx += 1

        # Get the current preprocessed chunk
        preprocessed_chunk = preprocessed_words[start_idx:chunk_end_idx]
        original_chunk = original_words[start_idx:chunk_end_idx]
        

        # Append chunks to results
        processed_chunks.append(" ".join([w for w in preprocessed_chunk if not w.startswith("<REMOVED:") and w != "<REMOVED>"]))
        original_chunks.append(" ".join([w for w in original_chunk]))

        # Move to the next chunk
        start_idx = chunk_end_idx

    # Handle last small chunk case
    if len(processed_chunks) > 1 and len(processed_chunks[-1].split()) < min_chunk_length:
        processed_chunks[-2] += " " + processed_chunks[-1]
        original_chunks[-2] += " " + original_chunks[-1]
        processed_chunks.pop()
        original_chunks.pop()

    return processed_chunks, original_chunks

# Apply the function to a DataFrame
def preprocess_and_chunk_dataframe(df, args):
    """Preprocess and chunk text in the DataFrame."""
    tqdm.pandas()
    df = df.drop_duplicates(subset=['text', args.id_column])
    # Preprocess text with placeholders
    df['unprocessed_text'] = df['text'].astype(str)
    df['text'] = df['text'].astype(str)
    df['text'].replace(to_replace=r"\.\.+", value=" ", regex=True, inplace=True)
    df['text'].replace(to_replace=r"\-\-+", value=" ", regex=True, inplace=True)
    df['text'].replace(to_replace=r"__+", value=" ", regex=True, inplace=True)
    df['text'].replace(to_replace=r"\*\*+", value=" ", regex=True, inplace=True)
    df['text'].replace(to_replace=r"\s+", value=" ", regex=True, inplace=True)

    df['text'] = df['text'].progress_apply(remove_special_characters_rm)
    df['text'] = df['text'].progress_apply(lambda x: remove_stopwords_rm(x, stopwords_set))
    df['length'] = df['text'].progress_apply(
                                            lambda text: len([
                                                word for word in 
                                                [re.sub(r'<REMOVED>', '', x) for x in text.split()] 
                                                if not word.startswith("<REMOVED:")
                                            ])
                                        )
    
    print(f"Length before dropping all speeches with less than {args.length_threshold} tokens: {len(df)}")
    df = df[df['length'] > args.length_threshold]
    print(f"Length after dropping all speeches with less than {args.length_threshold} tokens: {len(df)}")
    print(f"Average Speech length: {[df['length'].mean()]}")

    # Chunk text
    df['chunked_data'] = df.progress_apply(
        lambda row: chunk_preprocessed_and_original_with_mapping(row['text'], row['unprocessed_text'], args), axis=1
    )

    # Extract columns from chunked data
    df['text'] = df['chunked_data'].apply(lambda x: x[0])  # Processed text chunks
    df['original_chunks'] = df['chunked_data'].apply(lambda x: x[1])  # Original text chunks

    # Explode the DataFrame into rows for each chunk
    df = df.explode(['text', 'original_chunks'], ignore_index=True)

    # Drop intermediate column
    df.drop(columns=['chunked_data'], inplace=True)
    df = df.drop_duplicates(subset=['text']+[f'{args.id_column}'])
    df['text'] = df['text'].str.replace(r'<REMOVED>', '', regex=True)
    
    return df

In [5]:
df2.rename(columns={'sentence': 'text'}, inplace=True)

# Displaying the updated DataFrame to confirm the change

In [6]:
gc.collect()
df2 = preprocess_and_chunk_dataframe(df2,args)

/tmp/ipykernel_17/929610713.py:114: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['unprocessed_text'] = df['text'].astype(str)
/tmp/ipykernel_17/929610713.py:115: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].astype(str)
/tmp/ipykernel_17/929610713.py:116: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermedia

Length before dropping all speeches with less than 10 tokens: 8672377
Length after dropping all speeches with less than 10 tokens: 2700156
Average Speech length: [39.69238999524472]


100%|██████████| 2700156/2700156 [03:54<00:00, 11512.09it/s]


In [7]:
print(len(df2),len(df))

2972331 2972331


In [8]:
# Ensure the unique identifier column is set dynamically
id_column = f'{args.id_column}'  # Replace with your actual ID column name

# Map 'original_chunks' from df3 to df based on matching 'text' and unique identifier
df['original_text'] = df.set_index(['text', id_column]) \
                        .index.map(df2.set_index(['text', id_column])['original_chunks'])

# Identify rows where the mapping failed
missing_count = df['original_text'].isna().sum()

# Print the count of missing matches
print(f"Number of rows where the identifier was not found: {missing_count}")

Number of rows where the identifier was not found: 901


In [9]:

def sample_scores(df):
    sampled_df = pd.DataFrame()
    decades = df['decade'].unique()

    intuition_bins = df['intuition_bin'].unique()
    evidence_bins = df['evidence_bin'].unique()
    emi_bins = df['emi_bin'].unique()

    # Initialize counters for summary
    summary_counts = {
        "intuition_bin": {bin_: 0 for bin_ in intuition_bins},
        "evidence_bin": {bin_: 0 for bin_ in evidence_bins},
        "emi_bin": {bin_: 0 for bin_ in emi_bins},
    }

    # Keep track of already sampled text IDs across all bins
    sampled_text_ids = set()

    # Iterate through each decade
    for decade in decades:
        decade_df = df[(df['decade'] == decade) & (df['original_text'].notna())]  # Exclude rows where original_text is None or NaN

        # Calculate how many rows to sample from each bin category (half for each category)
        total_per_decade = 78  # Total samples per decade
        per_bin_sample = (total_per_decade // 3) // 4  # Half of the samples divided into 4 bins each for intuition and evidence

         # Sample for EMI bins
        for emi_bin in emi_bins:
            bin_df = decade_df[
                (decade_df['emi_bin'] == emi_bin) &
                (~decade_df['id'].isin(sampled_text_ids))
            ]
            bin_df = bin_df[(bin_df['chunk_length'] >= 20) & (bin_df['chunk_length'] <= 50)]  # Filter rows where chunk_length >= 20

            if len(bin_df) >= per_bin_sample:
                sampled_bin = bin_df.sample(per_bin_sample, random_state=4)
            else:
                sampled_bin = bin_df

            sampled_text_ids.update(sampled_bin['id'])
            summary_counts["emi_bin"][emi_bin] += len(sampled_bin)
            sampled_df = pd.concat([sampled_df, sampled_bin])
            
        # Sample for intuition bins
        for intuition_bin in intuition_bins:
            bin_df = decade_df[
                (decade_df['intuition_bin'] == intuition_bin) &
                (~decade_df['id'].isin(sampled_text_ids))
            ]
            bin_df = bin_df[(bin_df['chunk_length'] >= 20) & (bin_df['chunk_length'] <= 50)]  # Filter rows where chunk_length >= 20

            if len(bin_df) >= per_bin_sample:
                sampled_bin = bin_df.sample(per_bin_sample, random_state=4)
            else:
                sampled_bin = bin_df

            sampled_text_ids.update(sampled_bin['id'])
            summary_counts["intuition_bin"][intuition_bin] += len(sampled_bin)
            sampled_df = pd.concat([sampled_df, sampled_bin])

        # Sample for evidence bins
        for evidence_bin in evidence_bins:
            bin_df = decade_df[
                (decade_df['evidence_bin'] == evidence_bin) &
                (~decade_df['id'].isin(sampled_text_ids))
            ]
            bin_df = bin_df[(bin_df['chunk_length'] >= 20) & (bin_df['chunk_length'] <= 50)]  # Filter rows where chunk_length >= 20

            if len(bin_df) >= per_bin_sample:
                sampled_bin = bin_df.sample(per_bin_sample, random_state=4)
            else:
                sampled_bin = bin_df

            sampled_text_ids.update(sampled_bin['id'])
            summary_counts["evidence_bin"][evidence_bin] += len(sampled_bin)
            sampled_df = pd.concat([sampled_df, sampled_bin])


    # Print summary
    print("Sampling Summary:")
    print("Intuition Bins:")
    for bin_, count in summary_counts["intuition_bin"].items():
        print(f"  {bin_}: {count} texts")

    print("\nEvidence Bins:")
    for bin_, count in summary_counts["evidence_bin"].items():
        print(f"  {bin_}: {count} texts")

    print("\nEMI Bins:")
    for bin_, count in summary_counts["emi_bin"].items():
        print(f"  {bin_}: {count} texts")

    return sampled_df




In [10]:
# Converting the 'Date' column to datetime format
df['year'] = pd.to_datetime(df['date'])

# Extracting the year and replacing the 'Date' column with the year only
df['year'] = df['year'].dt.year

# Create a new column for decades
df['decade'] = (df['year'] // 10) * 10

# Define the bins based on the min and max of the evidence_minus_intuition_score
min_val = df['emi'].min() - 0.00001
max_val = df['emi'].max() + 0.00001
bins = pd.interval_range(start=min_val, end=max_val, periods=4)

# Create the bins and add them to the dataframe
df['emi_bin'] = pd.cut(df['emi'], bins=bins, labels=range(4))


# Create bins for intuition_z and evidence_z using interval ranges
min_intuition = df['intuition_z'].min() - 0.00001
max_intuition = df['intuition_z'].max() + 0.00001
intuition_bins = pd.interval_range(start=min_intuition, end=max_intuition, periods=4)
df['intuition_bin'] = pd.cut(df['intuition_z'], bins=intuition_bins)

min_evidence = df['evidence_z'].min() - 0.00001
max_evidence = df['evidence_z'].max() + 0.00001
evidence_bins = pd.interval_range(start=min_evidence, end=max_evidence, periods=4)
df['evidence_bin'] = pd.cut(df['evidence_z'], bins=evidence_bins)


sample_similarities = sample_scores(df)
sample_similarities

Sampling Summary:
Intuition Bins:
  (-1.7161776999999998, 0.46014310000000025]: 102 texts
  (-3.8924985, -1.7161776999999998]: 88 texts
  (0.46014310000000025, 2.6364639000000003]: 102 texts
  (2.6364639000000003, 4.8127847]: 98 texts

Evidence Bins:
  (-1.70590185, 1.1031212999999997]: 102 texts
  (1.1031212999999997, 3.9121444499999996]: 102 texts
  (-4.514925, -1.70590185]: 67 texts
  (3.9121444499999996, 6.721167599999999]: 85 texts

EMI Bins:
  (-2.1966578749999996, 0.79287625]: 102 texts
  (0.79287625, 3.7824103750000004]: 102 texts
  (-5.186191999999999, -2.1966578749999996]: 101 texts
  (3.7824103750000004, 6.7719445]: 91 texts


,id,begin,end,text,date,session,electoralTerm,firstName,lastName,politicianId,...,intuition_adj,evidence_z,intuition_z,emi,original_text,year,decade,emi_bin,intuition_bin,evidence_bin
1465877,1066202,49450.0,49817.0,Freunde Zugeständniß Anspruch genommen vornher...,1887-05-24,NaN,NaN,NaN,NaN,NaN,...,-0.065102,-0.385037,-0.558465,0.173428,"hat, auch für meine Freunde und mich das Zuges...",1887,1880,"(-2.1966578749999996, 0.79287625]","(-1.7161776999999998, 0.46014310000000025]","(-1.70590185, 1.1031212999999997]"
1437553,1062050,6033.0,6415.0,Dagegen namens politischen Freunde hervorzuheb...,1880-04-20,NaN,NaN,NaN,NaN,NaN,...,-0.109227,-1.687669,-0.936976,-0.750694,Dagegen habe ich namens meiner politischen Fre...,1880,1880,"(-2.1966578749999996, 0.79287625]","(-1.7161776999999998, 0.46014310000000025]","(-1.70590185, 1.1031212999999997]"
362240,1070246,91923.0,92226.0,glaube Produzenten Mosel Rhein verwahren ganz ...,1887-01-04,NaN,NaN,NaN,NaN,NaN,...,0.080756,-0.012401,0.692746,-0.705147,"Ich glaube, die Produzenten an der Mosel und a...",1887,1880,"(-2.1966578749999996, 0.79287625]","(0.46014310000000025, 2.6364639000000003]","(-1.70590185, 1.1031212999999997]"
1329383,1073366,129188.0,129952.0,endlich Bezug Worte oberste Landes finanzbehör...,1882-01-20,NaN,NaN,NaN,NaN,NaN,...,-0.181193,-1.455428,-1.554318,0.098890,Wenn endlich in Bezug auf die Worte „oberste L...,1882,1880,"(-2.1966578749999996, 0.79287625]","(-1.7161776999999998, 0.46014310000000025]","(-1.70590185, 1.1031212999999997]"
2080644,1070474,90728.0,91170.0,ganz besonders Worte gemeldet konstatiren alle...,1883-04-23,NaN,NaN,NaN,NaN,NaN,...,0.003213,0.231238,0.027565,0.203672,Ich habe mich ganz besonders zum Worte gemelde...,1883,1880,"(-2.1966578749999996, 0.79287625]","(-1.7161776999999998, 0.46014310000000025]","(-1.70590185, 1.1031212999999997]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2970396,1058090,NaN,NaN,ganz kurze Bemerkung mal darüber gesprochen wi...,2021-04-13,220.0,19.0,Martin,Neumann,11004120.0,...,0.401415,5.733139,3.443448,2.289691,Eine ganz kurze Bemerkung – wir hatten ja scho...,2021,2020,"(0.79287625, 3.7824103750000004]","(2.6364639000000003, 4.8127847]","(3.9121444499999996, 6.721167599999999]"
2954555,1035849,NaN,NaN,Nachfrage Verteidigungsausschuss Berichte Eins...,2020-01-29,142.0,19.0,Tobias,Lindner,11004217.0,...,0.118481,3.994513,1.016362,2.978151,Eine Nachfrage noch. Wenn wir uns im Verteidig...,2020,2020,"(0.79287625, 3.7824103750000004]","(0.46014310000000025, 2.6364639000000003]","(3.9121444499999996, 6.721167599999999]"
2956815,1038977,NaN,NaN,Herzlichen Dank Antworten weit zustimmen Daten...,2020-04-22,155.0,19.0,Andrew,Ullmann,11004922.0,...,0.133608,4.718647,1.146129,3.572518,Herzlichen Dank für die Antworten so weit. – W...,2020,2020,"(0.79287625, 3.7824103750000004]","(0.46014310000000025, 2.6364639000000003]","(3.9121444499999996, 6.721167599999999]"
2958095,1040845,NaN,NaN,möchte gerne antworten tut leid zugehört exakt...,2020-05-15,161.0,19.0,Bela,Bach,-1.0,...,0.224314,4.102027,1.924227,2.177800,"Ja, ich möchte gerne antworten. – Es tut mir l...",2020,2020,"(0.79287625, 3.7824103750000004]","(0.46014310000000025, 2.6364639000000003]","(3.9121444499999996, 6.721167599999999]"


In [11]:
test_evidence = "Die Daten unserer umfangreichen Studien zeigen klar, dass Investitionen in erneuerbare Energien nicht nur die CO₂-Emissionen bis 2030 um 40 % reduzieren können, sondern auch 250.000 neue Arbeitsplätze schaffen werden, wie aus dem Bericht des Bundesumweltamts hervorgeht."

test_intuition = "Wir müssen uns die Frage stellen, ob wir weiterhin auf kurzfristige Lösungen setzen oder ob wir den Mut haben, einen Weg zu gehen, von dem wir alle tief im Inneren wissen, dass er langfristig das Beste für unsere Gesellschaft ist."

In [12]:
# Define the test cases
test_data = [
    {
        'id': '0_testing',
        'text': "Die Daten unserer umfangreichen Studien zeigen klar, dass Investitionen in erneuerbare Energien nicht nur die CO₂-Emissionen bis 2030 um 40 % reduzieren können, sondern auch 250.000 neue Arbeitsplätze schaffen werden, wie aus dem Bericht des Bundesumweltamts hervorgeht.",
        'original_text': "Die Daten unserer umfangreichen Studien zeigen klar, dass Investitionen in erneuerbare Energien nicht nur die CO₂-Emissionen bis 2030 um 40 % reduzieren können, sondern auch 250.000 neue Arbeitsplätze schaffen werden, wie aus dem Bericht des Bundesumweltamts hervorgeht."
    },
    {
        'id': '1_testing',
        'text': "Wir müssen uns die Frage stellen, ob wir weiterhin auf kurzfristige Lösungen setzen oder ob wir den Mut haben, einen Weg zu gehen, von dem wir alle tief im Inneren wissen, dass er langfristig das Beste für unsere Gesellschaft ist.",
        'original_text': "Wir müssen uns die Frage stellen, ob wir weiterhin auf kurzfristige Lösungen setzen oder ob wir den Mut haben, einen Weg zu gehen, von dem wir alle tief im Inneren wissen, dass er langfristig das Beste für unsere Gesellschaft ist."
    }
]

# Create a DataFrame from the test data
test_df = pd.DataFrame(test_data)

# Ensure all columns from the original DataFrame are present in the test DataFrame
for column in sample_similarities.keys():
    if column not in test_df.columns:
        test_df[column] = None  # Set any missing column to NaN

# Append the test cases to the original DataFrame
sample_similarities = pd.concat([sample_similarities, test_df], ignore_index=True)


/tmp/ipykernel_17/919444143.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  sample_similarities = pd.concat([sample_similarities, test_df], ignore_index=True)
/tmp/ipykernel_17/919444143.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  sample_similarities = pd.concat([sample_similarities, test_df], ignore_index=True)
/tmp/ipykernel_17/919444143.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no long

In [13]:
sample_similarities

,id,begin,end,text,date,session,electoralTerm,firstName,lastName,politicianId,...,intuition_adj,evidence_z,intuition_z,emi,original_text,year,decade,emi_bin,intuition_bin,evidence_bin
0,1066202,49450.0,49817.0,Freunde Zugeständniß Anspruch genommen vornher...,1887-05-24,NaN,NaN,NaN,NaN,NaN,...,-0.065102,-0.385037,-0.558465,0.173428,"hat, auch für meine Freunde und mich das Zuges...",1887,1880,"(-2.1966578749999996, 0.79287625]","(-1.7161776999999998, 0.46014310000000025]","(-1.70590185, 1.1031212999999997]"
1,1062050,6033.0,6415.0,Dagegen namens politischen Freunde hervorzuheb...,1880-04-20,NaN,NaN,NaN,NaN,NaN,...,-0.109227,-1.687669,-0.936976,-0.750694,Dagegen habe ich namens meiner politischen Fre...,1880,1880,"(-2.1966578749999996, 0.79287625]","(-1.7161776999999998, 0.46014310000000025]","(-1.70590185, 1.1031212999999997]"
2,1070246,91923.0,92226.0,glaube Produzenten Mosel Rhein verwahren ganz ...,1887-01-04,NaN,NaN,NaN,NaN,NaN,...,0.080756,-0.012401,0.692746,-0.705147,"Ich glaube, die Produzenten an der Mosel und a...",1887,1880,"(-2.1966578749999996, 0.79287625]","(0.46014310000000025, 2.6364639000000003]","(-1.70590185, 1.1031212999999997]"
3,1073366,129188.0,129952.0,endlich Bezug Worte oberste Landes finanzbehör...,1882-01-20,NaN,NaN,NaN,NaN,NaN,...,-0.181193,-1.455428,-1.554318,0.098890,Wenn endlich in Bezug auf die Worte „oberste L...,1882,1880,"(-2.1966578749999996, 0.79287625]","(-1.7161776999999998, 0.46014310000000025]","(-1.70590185, 1.1031212999999997]"
4,1070474,90728.0,91170.0,ganz besonders Worte gemeldet konstatiren alle...,1883-04-23,NaN,NaN,NaN,NaN,NaN,...,0.003213,0.231238,0.027565,0.203672,Ich habe mich ganz besonders zum Worte gemelde...,1883,1880,"(-2.1966578749999996, 0.79287625]","(-1.7161776999999998, 0.46014310000000025]","(-1.70590185, 1.1031212999999997]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1139,1038977,NaN,NaN,Herzlichen Dank Antworten weit zustimmen Daten...,2020-04-22,155.0,19.0,Andrew,Ullmann,11004922.0,...,0.133608,4.718647,1.146129,3.572518,Herzlichen Dank für die Antworten so weit. – W...,2020,2020,"(0.79287625, 3.7824103750000004]","(0.46014310000000025, 2.6364639000000003]","(3.9121444499999996, 6.721167599999999]"
1140,1040845,NaN,NaN,möchte gerne antworten tut leid zugehört exakt...,2020-05-15,161.0,19.0,Bela,Bach,-1.0,...,0.224314,4.102027,1.924227,2.177800,"Ja, ich möchte gerne antworten. – Es tut mir l...",2020,2020,"(0.79287625, 3.7824103750000004]","(0.46014310000000025, 2.6364639000000003]","(3.9121444499999996, 6.721167599999999]"
1141,1053593,NaN,NaN,Konkrete Nachfrage gerade gesagt Kompromiss ma...,2021-01-27,205.0,19.0,Ulle,Schauws,11004395.0,...,0.155797,4.516490,1.336469,3.180022,Konkrete Nachfrage. Sie haben jetzt gerade ges...,2021,2020,"(0.79287625, 3.7824103750000004]","(0.46014310000000025, 2.6364639000000003]","(3.9121444499999996, 6.721167599999999]"
1142,0_testing,NaN,NaN,Die Daten unserer umfangreichen Studien zeigen...,None,NaN,NaN,None,None,NaN,...,NaN,NaN,NaN,NaN,Die Daten unserer umfangreichen Studien zeigen...,None,None,NaN,NaN,NaN


In [14]:
df.to_csv("speeches_all_emi.csv", index = False)
sample_similarities.to_csv("speeches_validation_sample.csv", index = False)

# Select only the 'id' and 'original_text' columns and rename 'original_text' to 'text'
sampled_subset = sample_similarities[['id', 'original_text']].rename(columns={'original_text': 'text'})

# Save the resulting DataFrame to a CSV file
sampled_subset.to_csv('survey_sample.csv', index=False)

In [15]:
# Ensure 'id' is treated as string for filtering
sampled_subset['id'] = sampled_subset['id'].astype(str)

# Extract mandatory rows (id = '0_testing' or '1_testing')
mandatory_rows = sampled_subset[sampled_subset['id'].isin(['0_testing', '1_testing'])]

# Remove these rows from the main DataFrame
remaining_rows = sampled_subset[~sampled_subset['id'].isin(['0_testing', '1_testing'])]

# Shuffle the remaining data
remaining_rows = remaining_rows.sample(frac=1, random_state=42).reset_index(drop=True)

# Split into 5 equal batches
batches = np.array_split(remaining_rows, 5)

# Add the mandatory rows to each batch
for i in range(5):
    batches[i] = pd.concat([mandatory_rows, batches[i]], ignore_index=True)
    # Save the batch to a CSV file
    batch_filename = f"survey_batch_{i+1}.csv"
    batches[i].to_csv(batch_filename, index=False)

/opt/conda/lib/python3.10/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
